In [ ]:
%run /Users/calvin.waldheim@gmail.com/lakebase_config

In [0]:
# Cell 1 - Install dependencies
%pip install psycopg2-binary databricks-sdk

In [0]:
with open("/Workspace/Users/calvin.waldheim@gmail.com/memory-scaling/concept.txt", "r") as f:
    SOURCE_TEXT = f.read()

print(f"Loaded {len(SOURCE_TEXT)} characters")

In [0]:
# Cell 3 - Chunk the document
def chunk_text(text, chunk_size=500, overlap=50):
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
        i += chunk_size - overlap
    return chunks

chunks = chunk_text(SOURCE_TEXT, chunk_size=150, overlap=20)
print(f"{len(chunks)} chunks created")

In [0]:
embeddings = []
for chunk in chunks:
    result = embed([chunk])
    embeddings.append(result[0])
    
print(f"{len(embeddings)} embeddings created")

In [0]:
# Cell 5 - Store in Lakebase
import psycopg2
import hashlib
import json


conn = psycopg2.connect(CONN_STRING, password=TOKEN)
cur = conn.cursor()

for chunk, embedding in zip(chunks, embeddings):
    content_hash = hashlib.md5(chunk.encode()).hexdigest()
    cur.execute("""
        INSERT INTO memories 
            (project_id, project_type, memory_type, scope, domain, rule, context, source_ref, content_hash, embedding, quality_score)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        ON CONFLICT DO NOTHING
    """, (
        "memory-kb-poc",
        "product",
        "episodic",
        "organizational",
        "architecture",
        chunk[:100],        # first 100 chars as the "rule" summary
        chunk,
        "concept-doc-v1",
        content_hash,
        json.dumps(embedding),
        0.8
    ))

conn.commit()
cur.close()
conn.close()
print(f"Stored {len(chunks)} memories.")

In [0]:
print(f"Doc length: {len(SOURCE_TEXT)} characters")
print(f"Chunks: {len(chunks)}")
print(chunks[0][:200])  # preview first chunk

In [0]:
# Delete all existing memories and re-bootstrap with smaller chunks
conn = psycopg2.connect(CONN_STRING, password=TOKEN)
cur = conn.cursor()
cur.execute("DELETE FROM memories WHERE project_id = 'memory-kb-poc';")
conn.commit()
cur.close()
conn.close()
print("Cleared.")